# Competitor Gap Analysis — Deep Sea
### Deep Sea — BERTopic + Full LLM Synthesis + Coverage Quality Assessment

Sam Torres — The SEO Mermaid

---

## Who this notebook is for
You have API keys, you have budget, and you want the most complete output.
This is the production-ready version of the pipeline.

## What you'll get
Everything from Open Water, plus:
- Coverage quality assessment per cluster (differentiated / authoritative / redundant)
- Cannibalization flags with within-cluster similarity scoring
- All clusters labeled — no budget cap
- SF pre-computed embeddings as an input option

## Input options
- **Option A** — Screaming Frog CSV (notebook generates embeddings)
- **Option B** — Screaming Frog with pre-computed embeddings (SF AI integration)
- **Option C** — Custom CSV

## Quick start
1. Upload your data
2. Fill in the CONFIGURATION cell
3. Runtime > Run all


## Step 0: Install dependencies

In [ ]:
!pip install sentence-transformers bertopic hdbscan scikit-learn pandas numpy matplotlib seaborn umap-learn -q
!pip install openai anthropic -q
print("Dependencies installed")

## Configuration — edit this cell before running

In [ ]:
# ── INPUT MODE ────────────────────────────────────────────────────────────────
# "csv"          — standard crawl CSV, notebook generates embeddings
# "sf_embeddings" — Screaming Frog pre-computed embeddings export
INPUT_MODE = "csv"

# FILE PATHS
YOUR_SITE_CSV  = "your_site.csv"
COMPETITOR_CSV = "competitor_site.csv"

# For sf_embeddings mode — path to SF embedding exports
# YOUR_SITE_EMBEDDINGS_CSV  = "your_site_embeddings.csv"
# COMPETITOR_EMBEDDINGS_CSV = "competitor_embeddings.csv"
# SF_URL_COLUMN             = "Address"
# SF_EMBEDDING_COLUMN       = "Embedding"   # Column containing the vector

# COLUMN MAPPING (csv mode)
URL_COLUMN   = "Address"
TEXT_COLUMNS = ["Title 1", "Meta Description 1", "H1-1"]

# LABELS
YOUR_SITE_LABEL  = "My Site"
COMPETITOR_LABEL = "Competitor"

# BERTOPIC
MIN_TOPIC_SIZE = 5
N_TOP_WORDS    = 8

# EMBEDDING MODEL (csv mode)
EMBEDDING_PROVIDER = "sentence_transformers"
ST_MODEL_NAME      = "all-MiniLM-L6-v2"
# OpenAI alternative:
# EMBEDDING_PROVIDER     = "openai"
# OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"
# OPENAI_EMBED_KEY       = "sk-..."

# LLM SYNTHESIS — full, no cap
LLM_PROVIDER     = "openai"
OPENAI_API_KEY   = ""
OPENAI_LLM_MODEL = "gpt-4o"
# Anthropic alternative:
# LLM_PROVIDER      = "anthropic"
# ANTHROPIC_API_KEY = ""
# ANTHROPIC_MODEL   = "claude-sonnet-4-6"

print("Configuration loaded")

## Step 1: Load and prepare data

In [ ]:
import pandas as pd
import numpy as np

def load_and_prep(filepath, url_col, text_cols, label):
    df = pd.read_csv(filepath, low_memory=False)
    df = df[df[url_col].notna() & (df[url_col].str.strip() != "")].copy()
    if "Content Type" in df.columns:
        df = df[df["Content Type"].str.contains("text/html", na=False)]
    if "Status Code" in df.columns:
        df = df[df["Status Code"] == 200]
    available_cols = [c for c in text_cols if c in df.columns]
    if not available_cols:
        raise ValueError(f"None of {text_cols} found in {filepath}. Check TEXT_COLUMNS config.")
    df["content"] = df[available_cols].fillna("").apply(
        lambda row: " | ".join(v for v in row if str(v).strip()), axis=1
    )
    df = df[df["content"].str.strip() != ""].copy()
    df = df[[url_col, "content"]].rename(columns={url_col: "url"})
    df["source"] = label
    df = df.reset_index(drop=True)
    print(f"  {label}: {len(df)} pages loaded")
    return df
def load_sf_embeddings(filepath, url_col, embedding_col):
    """Load pre-computed embeddings from a Screaming Frog export.
    SF exports embeddings as a JSON array string per row.
    """
    import ast
    df = pd.read_csv(filepath, low_memory=False)
    df = df[df[url_col].notna()].copy()
    df = df[df[embedding_col].notna()].copy()

    def parse_embedding(val):
        if isinstance(val, str):
            return np.array(ast.literal_eval(val), dtype=np.float32)
        return np.array(val, dtype=np.float32)

    embeddings = np.vstack(df[embedding_col].apply(parse_embedding).values)
    urls = df[url_col].tolist()
    print(f"  Loaded {len(urls)} pre-computed embeddings from Screaming Frog")
    print(f"  Embedding dimensions: {embeddings.shape[1]}")
    return urls, embeddings

if INPUT_MODE == "csv":
    print("Loading CSVs...")
    df_yours = load_and_prep(YOUR_SITE_CSV,  URL_COLUMN, TEXT_COLUMNS, YOUR_SITE_LABEL)
    df_comp  = load_and_prep(COMPETITOR_CSV, URL_COLUMN, TEXT_COLUMNS, COMPETITOR_LABEL)
    df_all   = pd.concat([df_yours, df_comp], ignore_index=True)
    sf_embeddings_loaded = False
    print(f"Total pages: {len(df_all)}")

elif INPUT_MODE == "sf_embeddings":
    print("Loading Screaming Frog pre-computed embeddings...")
    your_urls,  your_emb  = load_sf_embeddings(YOUR_SITE_EMBEDDINGS_CSV,  SF_URL_COLUMN, SF_EMBEDDING_COLUMN)
    comp_urls,  comp_emb  = load_sf_embeddings(COMPETITOR_EMBEDDINGS_CSV, SF_URL_COLUMN, SF_EMBEDDING_COLUMN)
    df_yours = pd.DataFrame({"url": your_urls, "content": "", "source": YOUR_SITE_LABEL})
    df_comp  = pd.DataFrame({"url": comp_urls, "content": "", "source": COMPETITOR_LABEL})
    df_all   = pd.concat([df_yours, df_comp], ignore_index=True)
    embeddings = np.vstack([your_emb, comp_emb])
    sf_embeddings_loaded = True
    print(f"Total pages: {len(df_all)} | Embedding dims: {embeddings.shape[1]}")

## Step 2: Generate embeddings

In [ ]:
def get_embeddings_st(texts, model_name):
    from sentence_transformers import SentenceTransformer
    print(f"Loading Sentence Transformers model: {model_name}")
    print("(First run downloads ~90MB — cached after that)")
    model = SentenceTransformer(model_name)
    print(f"Embedding {len(texts)} pages...")
    return model.encode(texts, show_progress_bar=True, batch_size=64)
# OpenAI embeddings — uncomment to use instead of Sentence Transformers
# def get_embeddings_openai(texts, api_key, model="text-embedding-3-small"):
#     from openai import OpenAI
#     import time
#     client = OpenAI(api_key=api_key)
#     all_embeddings = []
#     for i in range(0, len(texts), 100):
#         batch = texts[i:i+100]
#         response = client.embeddings.create(model=model, input=batch)
#         all_embeddings.extend([r.embedding for r in response.data])
#         print(f"  Embedded {min(i+100, len(texts))}/{len(texts)}")
#         time.sleep(0.5)
#     return np.array(all_embeddings)
# TIP: Save embeddings to disk to avoid re-running this step
# import numpy as np
# np.save("embeddings.npy", embeddings)
# To reload: embeddings = np.load("embeddings.npy")

if not sf_embeddings_loaded:
    texts = df_all["content"].tolist()
    if EMBEDDING_PROVIDER == "sentence_transformers":
        embeddings = get_embeddings_st(texts, ST_MODEL_NAME)
    elif EMBEDDING_PROVIDER == "openai":
        # embeddings = get_embeddings_openai(texts, OPENAI_EMBED_KEY, OPENAI_EMBEDDING_MODEL)
        raise ValueError("Uncomment get_embeddings_openai above to use OpenAI embeddings.")
    print(f"Embeddings shape: {embeddings.shape}")
else:
    print(f"Using pre-loaded SF embeddings: {embeddings.shape}")

## Step 3: BERTopic clustering

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

print(f"Running BERTopic (min_topic_size={MIN_TOPIC_SIZE})...")

umap_model      = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model   = HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, min_samples=1,
                            metric="euclidean", cluster_selection_method="eom", prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
topic_model      = BERTopic(umap_model=umap_model, hdbscan_model=hdbscan_model,
                             vectorizer_model=vectorizer_model, top_n_words=N_TOP_WORDS, verbose=True)

topics, _ = topic_model.fit_transform(
    df_all["content"].tolist() if INPUT_MODE == "csv" else [""] * len(df_all),
    embeddings
)
df_all["cluster"] = topics
topic_info     = topic_model.get_topic_info()
topic_name_map = dict(zip(topic_info["Topic"], topic_info["Name"]))
df_all["topic_name"] = df_all["cluster"].map(topic_name_map).fillna("Outlier")

print(f"Topics: {len(topic_info[topic_info['Topic'] != -1])} | Outliers: {len(df_all[df_all['cluster']==-1])}")
print(topic_info[topic_info["Topic"] != -1][["Topic","Count","Name"]].to_string(index=False))

## Step 4: Gap summary

In [ ]:
df_topics = df_all[df_all["cluster"] != -1].copy()
cluster_summary = (
    df_topics.groupby(["cluster", "topic_name", "source"])
    .size().unstack(fill_value=0).reset_index()
)
for col in [YOUR_SITE_LABEL, COMPETITOR_LABEL]:
    if col not in cluster_summary.columns:
        cluster_summary[col] = 0
cluster_summary["total"]     = cluster_summary[YOUR_SITE_LABEL] + cluster_summary[COMPETITOR_LABEL]
cluster_summary["gap_ratio"] = (
    cluster_summary[COMPETITOR_LABEL] / cluster_summary[YOUR_SITE_LABEL].replace(0, 0.1)
).round(2)
cluster_summary = cluster_summary.sort_values("gap_ratio", ascending=False)
print(cluster_summary[["cluster","topic_name",YOUR_SITE_LABEL,COMPETITOR_LABEL,"gap_ratio"]].to_string(index=False))

## Step 5: Visualize — Breadth & Depth

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Competitor Gap Analysis: Breadth & Depth", fontsize=16, fontweight="bold", y=1.02)
cs = cluster_summary.copy()
cs["short"] = cs["topic_name"].str[:28]
x, w = range(len(cs)), 0.35

ax1 = axes[0]
ax1.bar([i-w/2 for i in x], cs[YOUR_SITE_LABEL],  w, label=YOUR_SITE_LABEL,  color="#028090", alpha=0.85)
ax1.bar([i+w/2 for i in x], cs[COMPETITOR_LABEL], w, label=COMPETITOR_LABEL, color="#F96167", alpha=0.85)
ax1.set_xticks(list(x)); ax1.set_xticklabels(cs["short"], rotation=45, ha="right", fontsize=8)
ax1.set_ylabel("Number of Pages"); ax1.set_title("Depth: Pages per Cluster")
ax1.legend(); ax1.grid(axis="y", alpha=0.3)

ax2 = axes[1]
hm = cs[[YOUR_SITE_LABEL, COMPETITOR_LABEL]].copy(); hm.index = cs["short"]
sns.heatmap(hm.div(hm.max()).fillna(0).T, ax=ax2, cmap="YlOrRd", linewidths=0.5,
            annot=hm.T, fmt="g", annot_kws={"size": 8}, cbar_kws={"label": "Relative coverage"})
ax2.set_title("Breadth: Coverage Heatmap")
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig("competitor_gap_analysis.png", dpi=150, bbox_inches="tight")
plt.show(); print("Saved: competitor_gap_analysis.png")

## Step 6: Topic space map (UMAP)

In [ ]:
from umap import UMAP
import matplotlib.pyplot as plt

print("Running 2D UMAP for visualization (30-60s)...")
reducer_2d = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
coords = reducer_2d.fit_transform(embeddings)
df_all["umap_x"] = coords[:, 0]
df_all["umap_y"] = coords[:, 1]

fig, ax = plt.subplots(figsize=(13, 8))
site_colors  = {YOUR_SITE_LABEL: "#028090", COMPETITOR_LABEL: "#F96167"}
site_markers = {YOUR_SITE_LABEL: "o", COMPETITOR_LABEL: "^"}

outliers = df_all[df_all["cluster"] == -1] if -1 in df_all["cluster"].values else pd.DataFrame()
if len(outliers) > 0:
    ax.scatter(outliers["umap_x"], outliers["umap_y"],
               c="#CCCCCC", alpha=0.3, s=20, zorder=1, label="Outlier")

for source, group in df_all[df_all["cluster"] != -1].groupby("source"):
    ax.scatter(group["umap_x"], group["umap_y"],
               c=site_colors.get(source, "#888888"), marker=site_markers.get(source, "o"),
               alpha=0.65, s=45, label=source, zorder=2)

for cid, group in df_all[df_all["cluster"] != -1].groupby("cluster"):
    cx, cy = group["umap_x"].mean(), group["umap_y"].mean()
    label = str(topic_name_map.get(cid, f"T{cid}"))[:22]
    ax.annotate(label, (cx, cy), fontsize=8, fontweight="bold", color="#0D1B2A",
                bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.75, ec="#CCCCCC"))

ax.set_title("Topic Space Map: Your Site vs. Competitor", fontsize=14, fontweight="bold")
ax.set_xlabel("UMAP dimension 1")
ax.set_ylabel("UMAP dimension 2")
ax.legend(markerscale=1.5)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("topic_space_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: topic_space_map.png")


## Step 7: Collect cluster samples + within-cluster similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

SAMPLE_PER_CLUSTER = 6
cluster_samples = {}

for cluster_id in sorted(df_topics["cluster"].unique()):
    cluster_pages = df_topics[df_topics["cluster"] == cluster_id]
    topic_name    = topic_name_map.get(cluster_id, f"Topic {cluster_id}")
    sample        = cluster_pages.sample(min(SAMPLE_PER_CLUSTER, len(cluster_pages)), random_state=42)
    your_pages    = cluster_pages[cluster_pages["source"] == YOUR_SITE_LABEL]
    your_sample   = your_pages.sample(min(SAMPLE_PER_CLUSTER, len(your_pages)), random_state=42) if len(your_pages) > 0 else pd.DataFrame()
    topic_words   = topic_model.get_topic(cluster_id)
    keywords      = ", ".join([w for w, _ in topic_words[:6]]) if topic_words else "N/A"

    # Within-cluster similarity for your pages
    mean_sim, max_sim = None, None
    if len(your_pages) >= 2:
        idxs     = your_pages.index.tolist()
        emb_sub  = normalize(embeddings[idxs])
        sim_mat  = cosine_similarity(emb_sub)
        n        = len(idxs)
        upper    = [sim_mat[r, c] for r in range(n) for c in range(r+1, n)]
        mean_sim = round(float(sum(upper)/len(upper)), 4)
        max_sim  = round(float(max(upper)), 4)

    cluster_samples[cluster_id] = {
        "topic_name": topic_name, "keywords": keywords,
        "your_count": len(your_pages), "comp_count": len(cluster_pages[cluster_pages["source"] == COMPETITOR_LABEL]),
        "gap_ratio": round(len(cluster_pages[cluster_pages["source"] == COMPETITOR_LABEL]) / max(len(your_pages), 0.1), 2),
        "sample_content": sample["content"].tolist(),
        "your_sample_content": your_sample["content"].tolist() if len(your_sample) > 0 else [],
        "mean_sim": mean_sim, "max_sim": max_sim
    }

print(f"Collected samples for {len(cluster_samples)} clusters")

## Step 8: Full LLM Synthesis — gap analysis + coverage quality
No budget cap. All clusters. Coverage quality assessment included.
Each cluster gets: gap type + priority + recommended action + coverage quality classification.

In [ ]:
def call_llm(prompt, provider, api_key, model):
    if provider == "openai":
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        return r.choices[0].message.content.strip()
    elif provider == "anthropic":
        import anthropic
        client = anthropic.Anthropic(api_key=api_key)
        r = client.messages.create(
            model=model,
            max_tokens=600,
            messages=[{"role": "user", "content": prompt}]
        )
        return r.content[0].text.strip()
    else:
        raise ValueError(f"Unknown provider: {provider}")

import json, time

def build_full_prompt(cluster_id, data, your_label, comp_label):
    preview      = "\n".join([f"- {c[:180]}" for c in data["sample_content"][:5]])
    your_preview = "\n".join([f"- {c[:180]}" for c in data["your_sample_content"][:5]]) or "No pages from your site."
    sim_note     = ""
    if data["mean_sim"] is not None:
        sim_note = f"\nWithin-cluster similarity ({your_label} pages): mean={data['mean_sim']} max={data['max_sim']} (guide: <0.70 differentiated | 0.70-0.84 overlapping | 0.85+ likely redundant)"
    return f"""SEO strategist analyzing a BERTopic cluster.

CLUSTER {cluster_id}: {data['topic_name']}
Keywords: {data['keywords']}
{your_label}: {data['your_count']} pages | {comp_label}: {data['comp_count']} pages{sim_note}

All pages sample:
{preview}

{your_label} pages only:
{your_preview}

Return ONLY this JSON:
{{
  "cluster_topic": "2-5 word label",
  "description": "One sentence",
  "gap_type": "missing"|"thin"|"competitive"|"strong",
  "gap_type_explanation": "One sentence",
  "priority": "high"|"medium"|"low",
  "recommended_action": "One action",
  "coverage_quality": "differentiated"|"authoritative"|"redundant"|null,
  "coverage_quality_explanation": "One sentence or null",
  "coverage_action": "One action or null"
}}

Gap: missing=0/few your pages vs significant competitor | thin=some but competitor 2x+ | competitive=similar | strong=you equal/greater
Coverage: differentiated=distinct angles healthy | authoritative=real depth minimal overlap | redundant=near-duplicate cannibalization risk
If {your_label} has 0 pages set coverage_quality and coverage_action to null.
Return ONLY the JSON."""


api_key = OPENAI_API_KEY if LLM_PROVIDER == "openai" else globals().get("ANTHROPIC_API_KEY", "")
model   = OPENAI_LLM_MODEL if LLM_PROVIDER == "openai" else globals().get("ANTHROPIC_MODEL", "")
results = []

if not api_key:
    print("No API key — skipping LLM synthesis.")
else:
    print(f"Running full synthesis on all {len(cluster_samples)} clusters...")
    for cluster_id, data in cluster_samples.items():
        print(f"  [{cluster_id}] {data['topic_name'][:40]}...", end=" ")
        try:
            raw    = call_llm(build_full_prompt(cluster_id, data, YOUR_SITE_LABEL, COMPETITOR_LABEL),
                              LLM_PROVIDER, api_key, model)
            parsed = json.loads(raw)
            parsed.update({
                "cluster_id": cluster_id, "bertopic_name": data["topic_name"],
                "keywords": data["keywords"], "your_pages": data["your_count"],
                "comp_pages": data["comp_count"], "mean_similarity": data["mean_sim"],
                "max_similarity": data["max_sim"]
            })
            results.append(parsed)
            q = parsed.get("coverage_quality", "—")
            print(f"✅ {parsed['cluster_topic']} | {q}")
        except Exception as e:
            print(f"Error: {e}")
        time.sleep(0.4)
    print(f"\nComplete: {len(results)} clusters labeled")

## Step 9: Full gap report with coverage quality flags

In [ ]:
q_emoji = {"differentiated": "✅", "authoritative": "⭐", "redundant": "⚠️ ", None: "—"}

if not api_key or not results:
    cluster_summary.to_csv("competitor_gap_summary.csv", index=False)
    print("No LLM results. Cluster summary saved.")
else:
    report_df = pd.DataFrame(results)
    report_df["ps"] = report_df["priority"].map({"high":0,"medium":1,"low":2})
    report_df["gs"] = report_df["gap_type"].map({"missing":0,"thin":1,"competitive":2,"strong":3})
    report_df = report_df.sort_values(["ps","gs"]).drop(columns=["ps","gs"])

    print("\n" + "="*70)
    print("COMPETITOR GAP REPORT — FULL ANALYSIS")
    print("="*70 + "\n")

    for _, row in report_df.iterrows():
        g  = {"high":"🔴","medium":"🟡","low":"🟢"}.get(row["priority"],"⚪")
        q  = row.get("coverage_quality")
        qe = q_emoji.get(q, "—")
        ms = f" (mean sim: {row.get('mean_similarity')})" if row.get("mean_similarity") else ""
        print(f"{g} [{row['priority'].upper()}] {row['cluster_topic']}")
        print(f"   {row['description']}")
        print(f"   Gap: {row['gap_type'].upper()} — {row['gap_type_explanation']}")
        print(f"   Pages: {YOUR_SITE_LABEL}: {row['your_pages']} | {COMPETITOR_LABEL}: {row['comp_pages']}")
        print(f"   Action: {row['recommended_action']}")
        if q:
            print(f"   Coverage: {qe} {(q or '').upper()}{ms} — {row.get('coverage_quality_explanation','')}")
            if row.get("coverage_action"):
                print(f"   Coverage action: {row['coverage_action']}")
        print()

    redundant = report_df[report_df["coverage_quality"] == "redundant"]
    if len(redundant) > 0:
        print("─"*70)
        print(f"⚠️  REDUNDANCY FLAGS: {len(redundant)} cluster(s) — potential cannibalization")
        for _, row in redundant.iterrows():
            ms = f" | mean sim: {row.get('mean_similarity')}" if row.get("mean_similarity") else ""
            print(f"   {row['cluster_topic']}{ms}")
        print()

    cols = ["cluster_id","cluster_topic","bertopic_name","keywords","your_pages","comp_pages",
            "gap_type","priority","recommended_action","coverage_quality",
            "coverage_quality_explanation","coverage_action","mean_similarity","max_similarity"]
    report_df[[c for c in cols if c in report_df.columns]].to_csv("competitor_gap_report.csv", index=False)
    df_all[["url","source","cluster","topic_name"]].to_csv("all_pages_clustered.csv", index=False)
    print("Saved: competitor_gap_report.csv, all_pages_clustered.csv")

---
## You're done!

### Output files
| File | What it contains |
|------|-----------------|
| `competitor_gap_analysis.png` | Breadth & depth charts |
| `topic_space_map.png` | UMAP topic landscape |
| `competitor_gap_report.csv` | Full report: gap + coverage quality + similarity scores |
| `all_pages_clustered.csv` | Every page with cluster assignment |

### Reading coverage quality
| Flag | Emoji | Meaning | Action |
|------|-------|---------|--------|
| differentiated | ✅ | Distinct angles — healthy | Maintain |
| authoritative | ⭐ | Real depth, minimal overlap | Protect and build on |
| redundant | ⚠️ | Near-duplicate, cannibalization risk | Consolidate or differentiate |

---
*Sam Torres — The SEO Mermaid*
